# How to solve any text based AI task in a few lines of code with Kiri
> "Use transfer learning with Kiri to solve tasks with text generation"

- toc: false
- badges: true
- comments: true

Models are the backbone of AI. When trained and used correctly, they can solve real world tasks such as face recognition, email sorting and many more.

Until recently, most models were trained on a narrow set of data. This meant that you needed to train a new model from scratch for every single niche task that you wanted to solve.

With the race to General AI, models are becoming more general and thus more approachable. As they model more general data, they can solve a broader range of tasks.

Now, anyone with some basic programming skills can take a general model and solve their niche tasks with finetuning.

## Finetuning

Finetuning lets you take a model that has been trained on a very broad task and adapt it to your specific niche.

For general models this works really well. To intuitively understand why, let's look at a text generation model.

T5, an open-source model by Google, has been trained on around 750GB of text. The task was simply to predict some masked words within sentences. Take this sentence as an example: "The man went to the `___`, he bought a gallon of `___`." In order to correctly fill in the gaps, the model must understand enough about language and the world.

It turns out that this knowledge is transferrable to other tasks, hence the term transfer learning.

## Text generation with Kiri

Kiri is built around transfer learning for general models. Text generation with T5, using our [open-source library](https://github.com/kiri-ai/kiri), is an excellent example of this.

Kiri provides a single model that can do question answering, text summarisation and emotion detection.

In [1]:
import kiri
from kiri.models import T5QASummaryEmotion

t5 = T5QASummaryEmotion()

qa = kiri.tasks.QA(t5)
emotion = kiri.tasks.Emotion(t5)
summarisation = kiri.tasks.Summarisation(t5)

In [2]:
context = """In 2002, Musk founded SpaceX, an aerospace manufacturer and space transport services company, of which he is CEO, CTO, and lead designer."""
qa("What is SpaceX?", context)

'aerospace manufacturer and space transport services company'

In [3]:
emotion("Holy crap! This is awesome!")

'admiration'

In [4]:
long_text = """Artificial intelligence was founded as an academic discipline in 1955, and in the years since has experienced several waves of optimism,[13][14] followed by disappointment and the loss of funding (known as an "AI winter"),[15][16] followed by new approaches, success and renewed funding.[14][17] After AlphaGo successfully defeated a professional Go player in 2015, artificial intelligence once again attracted widespread global attention.[18] For most of its history, AI research has been divided into sub-fields that often fail to communicate with each other.[19] These sub-fields are based on technical considerations, such as particular goals (e.g. "robotics" or "machine learning"),[20] the use of particular tools ("logic" or artificial neural networks), or deep philosophical differences.[23][24][25] Sub-fields have also been based on social factors (particular institutions or the work of particular researchers).[19]

The traditional problems (or goals) of AI research include reasoning, knowledge representation, planning, learning, natural language processing, perception and the ability to move and manipulate objects.[20] General intelligence is among the field's long-term goals.[26] Approaches include statistical methods, computational intelligence, and traditional symbolic AI. Many tools are used in AI, including versions of search and mathematical optimization, artificial neural networks, and methods based on statistics, probability and economics. The AI field draws upon computer science, information engineering, mathematics, psychology, linguistics, philosophy, and many other fields.

The field was founded on the assumption that human intelligence "can be so precisely described that a machine can be made to simulate it".[27] This raises philosophical arguments about the mind and the ethics of creating artificial beings endowed with human-like intelligence. These issues have been explored by myth, fiction and philosophy since antiquity.[32] Some people also consider AI to be a danger to humanity if it progresses unabated.[33][34] Others believe that AI, unlike previous technological revolutions, will create a risk of mass unemployment.[35]"""

summarisation(long_text)

'Artificial intelligence was founded as an academic discipline in 1955. It was founded on the assumption that human intelligence can be "simplified" Some people consider AI to be a danger to humanity if it progresses unabated.'

The fact that a single model can accomplish these diverse tasks is amazing!

With some lines of code, you too can benefit from transfer learning and solve your niche tasks.

## Generating questions

Assume that you are making an app that makes it easier for teachers to make reading comprehension tests.

You need a model that can take some text and produce questions that can be answered based on that text.

Text generation takes some text as input and produces some text as output. As long as you can frame your problem as a text generation problem then anything is possible.

<img src="texttask.png">

This is exactly what Kiri's T5 model does to answer questions, detect emotion and summarise text.

### Preparing the data

Data is still needed to finetune a model. The good news is that you need *a lot* less of it. For our proof of concept, we will use less than 1000 examples.

Our examples are from the SQuAD dataset.

This dataset has multiple questions and answers on different paragraphs of text. What we'll do is get a paragraph of text as input and list of questions as output. Pretty simple!

In [5]:
#hide
from datasets import load_dataset

dataset = load_dataset("squad")

Reusing dataset squad (/home/kristo/.cache/huggingface/datasets/squad/plain_text/1.0.0/1244d044b266a5e4dbd4174d23cb995eead372fbca31a03edc3f8a132787af41)


In [6]:
#hide
dataset["train"][0]

{'answers': {'answer_start': [515], 'text': ['Saint Bernadette Soubirous']},
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'id': '5733be284776f41900661182',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'title': 'University_of_Notre_Dame'}

In [7]:
#hide
input_data = []
output_data = []

last_context = ""

# Limit to 5000 items for proof of concept
for i in range(5000):
    context = dataset["train"][i]["context"]
    question = dataset["train"][i]["question"]
    if context != last_context:
        input_data.append(context)
        last_context = context
        output_data.append([])

    output_index = len(input_data) - 1
    output_data[output_index].append(question)

In [8]:
input_data[0]

'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [9]:
output_data[0]

['To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'What is in front of the Notre Dame Main Building?',
 'The Basilica of the Sacred heart at Notre Dame is beside to which structure?',
 'What is the Grotto at Notre Dame?',
 'What sits on top of the Main Building at Notre Dame?']

In [10]:
len(input_data), len(output_data)

(820, 820)

We have taken 820 examples for training. That's just a small portion of the data, but it's enough to achieve some promising results. There is just 1 final step before finetuning.

The T5 model that we are planning on using has already been finetuned on some tasks such as translation. For multiple tasks, it is useful to add a prefix to the input that can let the model know what it should do.

The T5 model shown earlier differentiated between emotion detection and summarisation by prepending `emotion: ` and `summarise: ` respectively.

For our use case, we could use `generate questions: ` as the prefix.

For example: `generate questions: Some paragraph of text`.

Additionally, our `output_data` is currently a list of strings. Our model expects a single string for an output example.

In [11]:
input_data = [f"generate questions: {i}" for i in input_data]
output_data = ["; ".join(o) for o in output_data]

In [12]:
input_data[0]

'generate questions: Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [13]:
output_data[0]

'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?; What is in front of the Notre Dame Main Building?; The Basilica of the Sacred heart at Notre Dame is beside to which structure?; What is the Grotto at Notre Dame?; What sits on top of the Main Building at Notre Dame?'

For output data, we chose `;` as a separator that's not too common in text. Anything similar should work fine.

### Finetuning

We are ready to finetune. All we'll need to do is pass the list of input and output strings to Kiri.

In [14]:
#hide_output
# Start a local text generation task with T5
tg = kiri.tasks.TextGeneration(kiri.models.T5, local=True)
# Length here refers to number of tokens (1 token ~ 1 word)
tg.finetune(input_data, output_data, max_input_length=256, max_output_length=256, epochs=8)

Processing data...


GPU available: True, used: True
TPU available: None, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/kristo/.local/lib/python3.8/site-packages/pytorch_lightning/utilities/distributed.py:49: UserWarning: Field `model.batch_size` and `model.hparams.batch_size` are mutually exclusive! `model.batch_size` will be used as the initial batch size for scaling. If this is not the intended behavior, please remove either one.
  warnings.warn(*args, **kwargs)


Finding the optimal batch size...


/home/kristo/.local/lib/python3.8/site-packages/transformers/optimization.py:557: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  /pytorch/torch/csrc/utils/python_arg_parser.cpp:882.)
  exp_avg_sq_row.mul_(beta2t).add_(1.0 - beta2t, update.mean(dim=-1))
Batch size 2 succeeded, trying batch size 4
Batch size 4 succeeded, trying batch size 8
Batch size 8 succeeded, trying batch size 16
Batch size 16 succeeded, trying batch size 32
Batch size 32 succeeded, trying batch size 64
Batch size 64 succeeded, trying batch size 128
Batch size 128 failed, trying batch size 64
Finished batch size finder, will continue with full run using batch size 64
GPU available: True, used: True
TPU available: None, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type                       | Params
------------------------------------

Starting to train...


...

Validating: |          | 0/? [00:00<?, ?it/s]

Validating: |          | 0/? [00:00<?, ?it/s]

Validating: |          | 0/? [00:00<?, ?it/s]

Validating: |          | 0/? [00:00<?, ?it/s]

Training finished! Save your model for later with kiri.save or upload it with kiri.upload


In [15]:
input_text = """generate questions: Tesla, Inc. (originally Tesla Motors) was incorporated in July 2003 by Martin Eberhard and Marc Tarpenning, who financed the company until the Series A round of funding.[90] Both men played active roles in the company's early development prior to Musk's involvement.[91] Musk led the Series A round of investment in 2004, joining Tesla's board of directors as its chairman.[92][93][94][95] Musk took an active role within the company and oversaw Roadster product design but was not deeply involved in day-to-day business operations.[96] Following a series of escalating conflicts in 2007 and the 2008 financial crisis, Eberhard was ousted from the firm.[97][98] Musk assumed leadership of the company as CEO and product architect in 2008, positions he still holds today. A 2009 lawsuit settlement with Eberhard designated Musk as a Tesla co-founder, along with Tarpenning and two others.[4][5] As of 2019, Musk is the longest tenured CEO of any automotive manufacturer globally.[99]"""

In [16]:
# Have a look at Kiri's documentation to understand these parameters
tg(input_text, max_length=256, min_length=50, temperature=0.5)

'In what year was Tesla founded?; In what year was the Series A round of investment underway?; What is the name of the company that is named after Eberhard and who?; Who is the longest tenured CEO of any manufacturer?; What year was'

Not bad!

We used fewer than 1000 training examples, and less than 1 minute for training. This is the power of transfer learning!

## Next steps

If you have any task at all you need to solve, try to frame it as a text generation task.

Even if you do not have a lot of data, try to follow the steps in this tutorial - you may get some very promising results!

You can save this model for later with `kiri.save(model)` and load with `model = kiri.load("your-model-name")`.

If you want to use this model in an actual application, you will need to productionise it. Kiri supports deploying your model to an API with `kiri.upload(model, api_key="abc")`


Share your results with us and check out Kiri's [model platform](https://kiri.ai) and [library](https://github.com/kiri-ai/kiri) for more!